# PoC - Modelo de Status de Trafego

Este notebook cria uma prova de conceito para classificar o `traffic_status`
a partir dos dados de radar.

Regra de rotulo:
- `congestionado`: velocidade_aferida / velocidade_via <= 0.40
- `moderado`: 0.40 < razao <= 0.70
- `livre`: razao > 0.70


In [6]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

DATA_DIR = Path('/home/makleyston/Projects/service-composition/producer/radar/dataset/20230901')
MODEL_PATH = Path('model.joblib')
SAMPLE_SIZE = 300_000
RANDOM_STATE = 42


In [7]:
files = sorted(DATA_DIR.glob('*.json'))
if not files:
    raise FileNotFoundError(f'Nenhum arquivo JSON encontrado em {DATA_DIR}')

frames = [pd.read_json(file) for file in files]
df = pd.concat(frames, ignore_index=True)

print(f'Arquivos lidos: {len(files)}')
print(f'Registros totais: {len(df):,}')
df.head()


Arquivos lidos: 24
Registros totais: 2,470,755


,ID EQP,DATA HORA,MILESEGUNDO,FAIXA,ID DE ENDEREÇO,VELOCIDADE DA VIA,VELOCIDADE AFERIDA,CLASSIFICAÇÃO,TAMANHO,NUMERO DE SÉRIE,LATITUDE,LONGITUDE,ENDEREÇO,SENTIDO
0,195,2023-09-01T00:00:00,951,2,383,60,50.0,MOTO,1.0,3266,-19.90500,-43.93572,"Av. Cristiano Machado, a 12 metros da rua Pita...",Centro/Bairro
1,201,2023-09-01T00:00:00,372,3,398,60,50.0,MOTO,1.0,3276,-19.86400,-43.93149,"Av. Cristiano Machado, a 89 metros da Rua Cajuí",Bairro/Centro
2,254,2023-09-01T00:00:00,591,2,406,60,54.0,AUTOMÓVEL,3.2,2430,-19.92411,-43.98257,Av. Presidente Juscelino Kubitschek nº4333,Bairro/Centro
3,242,2023-09-01T00:00:00,281,2,336,70,50.0,AUTOMÓVEL,3.6,3284,-19.87740,-43.92909,"Av. Cristiano Machado, nº 9.950",Centro/Bairro
4,223,2023-09-01T00:00:00,482,4,347,60,30.0,AUTOMÓVEL,3.8,3296,-19.96350,-43.96328,"Av. Barão Homem de Melo, nº 3.191",Av. Raja Gabáglia / Av. Silva Lobo


In [8]:
rename_map = {
    'ID EQP': 'id_eqp',
    'DATA HORA': 'data_hora',
    'MILESEGUNDO': 'milisegundo',
    'FAIXA': 'faixa',
    'ID DE ENDEREÇO': 'id_endereco',
    'VELOCIDADE DA VIA': 'velocidade_via',
    'VELOCIDADE AFERIDA': 'velocidade_aferida',
    'CLASSIFICAÇÃO': 'classificacao',
    'TAMANHO': 'tamanho',
    'NUMERO DE SÉRIE': 'numero_serie',
    'LATITUDE': 'latitude',
    'LONGITUDE': 'longitude',
    'ENDEREÇO': 'endereco',
    'SENTIDO': 'sentido',
}

df = df.rename(columns=rename_map)

numeric_cols = [
    'id_eqp',
    'milisegundo',
    'faixa',
    'id_endereco',
    'velocidade_via',
    'velocidade_aferida',
    'latitude',
    'longitude',
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['data_hora'] = pd.to_datetime(df['data_hora'], errors='coerce')
df = df.dropna(subset=['data_hora', 'velocidade_via', 'velocidade_aferida'])
df = df[df['velocidade_via'] > 0]

df['hour'] = df['data_hora'].dt.hour
df['weekday'] = df['data_hora'].dt.weekday
df['speed_ratio'] = df['velocidade_aferida'] / df['velocidade_via']

df['traffic_status'] = np.select(
    [
        df['speed_ratio'] <= 0.40,
        (df['speed_ratio'] > 0.40) & (df['speed_ratio'] <= 0.70),
        df['speed_ratio'] > 0.70,
    ],
    [
        'congestionado',
        'moderado',
        'livre',
    ],
    default='moderado',
)

df['traffic_status'].value_counts(normalize=True)


traffic_status
livre            0.594411
moderado         0.315414
congestionado    0.090176
Name: proportion, dtype: float64

In [9]:
if len(df) > SAMPLE_SIZE:
    df_model, _ = train_test_split(
        df,
        train_size=SAMPLE_SIZE,
        random_state=RANDOM_STATE,
        stratify=df['traffic_status'],
    )
else:
    df_model = df.copy()

feature_cols = [
    'id_eqp',
    'faixa',
    'id_endereco',
    'velocidade_via',
    'classificacao',
    'tamanho',
    'sentido',
    'latitude',
    'longitude',
    'hour',
    'weekday',
]
target_col = 'traffic_status'

X = df_model[feature_cols]
y = df_model[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

categorical_features = ['classificacao', 'tamanho', 'sentido']
numeric_features = [c for c in feature_cols if c not in categorical_features]

X_train.shape, X_test.shape


((240000, 11), (60000, 11))

In [10]:
numeric_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore')),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features),
    ]
)

clf = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        (
            'model',
            RandomForestClassifier(
                n_estimators=160,
                min_samples_leaf=2,
                class_weight='balanced_subsample',
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))
print('Matriz de confusao:')
print(confusion_matrix(y_test, y_pred, labels=['congestionado', 'moderado', 'livre']))


               precision    recall  f1-score   support

congestionado       0.40      0.71      0.51      5410
        livre       0.81      0.74      0.77     35665
     moderado       0.53      0.50      0.51     18925

     accuracy                           0.66     60000
    macro avg       0.58      0.65      0.60     60000
 weighted avg       0.69      0.66      0.67     60000

Matriz de confusao:
[[ 3817  1142   451]
 [ 3827  9499  5599]
 [ 2011  7383 26271]]


In [11]:
preview = X_test.head(5).copy()
preview['status_real'] = y_test.head(5).values
preview['status_predito'] = clf.predict(X_test.head(5))
preview


,id_eqp,faixa,id_endereco,velocidade_via,classificacao,tamanho,sentido,latitude,longitude,hour,weekday,status_real,status_predito
2180831,253,1,405,60,MOTO,1.0,Bairro/Centro,-19.95539,-43.94042,20,4,livre,moderado
990197,222,3,310,60,AUTOMÓVEL,4.7,Bairro/Centro,-19.93030,-43.96205,12,4,livre,livre
436523,210,4,337,60,AUTOMÓVEL,2.8,Centro/Bairro,-19.92670,-43.95242,8,4,moderado,moderado
1646096,201,2,398,60,AUTOMÓVEL,5.0,Bairro/Centro,-19.86400,-43.93149,16,4,livre,livre
837653,220,1,344,70,AUTOMÓVEL,4.3,Bairro/Centro,-19.81910,-43.94702,11,4,livre,moderado


In [12]:
artifact = {
    'pipeline': clf,
    'features': feature_cols,
    'target': target_col,
    'labeling_rule': {
        'congestionado': 'velocidade_aferida / velocidade_via <= 0.40',
        'moderado': '0.40 < velocidade_aferida / velocidade_via <= 0.70',
        'livre': 'velocidade_aferida / velocidade_via > 0.70',
    },
    'metadata': {
        'sample_size': int(len(df_model)),
        'train_size': int(len(X_train)),
        'test_size': int(len(X_test)),
        'created_at_utc': pd.Timestamp.utcnow().isoformat(),
    },
}

joblib.dump(artifact, MODEL_PATH)
print(f'Modelo salvo em: {MODEL_PATH.resolve()}')


/tmp/ipykernel_190281/3060822162.py:14: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  'created_at_utc': pd.Timestamp.utcnow().isoformat(),


Modelo salvo em: /home/makleyston/Projects/service-composition/services/L0/traffic-speed/model.joblib
